# MusicGen — Text-to-Music Generation

Meta's MusicGen generates music from text descriptions. It uses:
- **Text encoder**: T5 encodes text prompts
- **Audio language model**: Transformer generates audio tokens
- **EnCodec decoder**: Converts tokens back to audio

We use `musicgen-small` (300M params) which runs on Colab free tier T4 GPU.

In [ ]:
!pip install transformers torch torchaudio scipy matplotlib IPython

In [ ]:
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import torch
import scipy
import IPython.display as ipd
import matplotlib.pyplot as plt
import numpy as np

processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small")
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
print(f"Model loaded on {next(model.parameters()).device}")
print(f"Sampling rate: {model.config.audio_encoder.sampling_rate} Hz")

## Text-to-Music Generation

In [ ]:
prompts = [
    "An upbeat electronic dance track with a driving bassline",
    "A calm acoustic guitar melody with birds singing",
    "Dark ambient drone music with deep bass and reverb",
]

inputs = processor(text=prompts, padding=True, return_tensors="pt").to(model.device)

with torch.no_grad():
    audio_values = model.generate(**inputs, max_new_tokens=512)  # ~10 seconds

# Play each result
sampling_rate = model.config.audio_encoder.sampling_rate
for i, (prompt, audio) in enumerate(zip(prompts, audio_values)):
    print(f"\nPrompt: {prompt}")
    ipd.display(ipd.Audio(audio.cpu().numpy(), rate=sampling_rate))

## Controlling Generation Length

The `max_new_tokens` parameter controls the duration of generated audio.
MusicGen's EnCodec operates at 50 tokens/second, so:
- 256 tokens = ~5 seconds
- 512 tokens = ~10 seconds
- 1024 tokens = ~20 seconds

In [ ]:
prompt = "A bright jazz piano trio with upright bass and brushes"
inputs = processor(text=[prompt], padding=True, return_tensors="pt").to(model.device)

durations = {"5 seconds": 256, "10 seconds": 512, "20 seconds": 1024}

for label, tokens in durations.items():
    print(f"\nGenerating {label} ({tokens} tokens)...")
    with torch.no_grad():
        audio = model.generate(**inputs, max_new_tokens=tokens)
    actual_duration = audio.shape[-1] / sampling_rate
    print(f"Actual duration: {actual_duration:.1f}s")
    ipd.display(ipd.Audio(audio[0].cpu().numpy(), rate=sampling_rate))

## Experimenting with Prompts

The quality of generation depends heavily on prompt design. Try:
- Genre + instruments + mood
- Specific musical attributes (tempo, key, dynamics)
- References to styles or eras

In [ ]:
# Try your own prompt here
my_prompt = "Jazz piano trio with walking bass and brushes on drums"

inputs = processor(text=[my_prompt], padding=True, return_tensors="pt").to(model.device)
with torch.no_grad():
    audio = model.generate(**inputs, max_new_tokens=1024)

print(f"Prompt: {my_prompt}")
print(f"Duration: {audio.shape[-1] / sampling_rate:.1f}s")
ipd.display(ipd.Audio(audio[0].cpu().numpy(), rate=sampling_rate))

## Visualize Generated Audio

In [ ]:
# Visualize waveform and spectrogram of the last generation
audio_np = audio[0].cpu().numpy().squeeze()

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Waveform
time = np.arange(len(audio_np)) / sampling_rate
axes[0].plot(time, audio_np, linewidth=0.5)
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")
axes[0].set_title(f"Waveform: {my_prompt}")

# Spectrogram
axes[1].specgram(audio_np, Fs=sampling_rate, NFFT=2048, noverlap=1024, cmap="magma")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Frequency (Hz)")
axes[1].set_title("Spectrogram")
axes[1].set_ylim(0, 8000)

plt.tight_layout()
plt.show()

## Save Outputs

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)

# Save the generated audio as WAV files
for i, (prompt, audio_tensor) in enumerate(zip(prompts, audio_values)):
    audio_np = audio_tensor.cpu().numpy().squeeze()
    # Normalize to 16-bit range
    audio_int16 = (audio_np * 32767).astype(np.int16)
    filename = f"outputs/musicgen_output_{i+1}.wav"
    scipy.io.wavfile.write(filename, sampling_rate, audio_int16)
    safe_prompt = prompt[:50].replace(' ', '_')
    print(f"Saved {filename} — '{prompt[:60]}...'")

print("\nAll outputs saved to outputs/ directory.")